# V036 confirmation contract proof

Synthetic full-architecture fixed fits for every member, representative exact resume, label filtering and prelabel-lock order. Full-capacity CUDA timing gates both standard-lease stages. No approved-data pressure values are read by these tests.

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import copy,json,hashlib,inspect,tempfile,csv,gc
import numpy as np
import torch

def run_confirmation_preflight_v036(project_root,output_directory,device='cpu',deep_spec=None):
    P=Path(project_root).resolve();O=Path(output_directory);O.mkdir(parents=True,exist_ok=False);checks={}
    def check(name,c):
        if not bool(c):raise AssertionError(name)
        checks[name]=True
    def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
    def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
    def equal(a,b):
        if torch.is_tensor(a):return torch.is_tensor(b) and torch.equal(a,b)
        if isinstance(a,np.ndarray):return isinstance(b,np.ndarray) and np.array_equal(a,b)
        if isinstance(a,dict):return a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
        if isinstance(a,(list,tuple)):return len(a)==len(b) and all(equal(x,y) for x,y in zip(a,b))
        return a==b
    choice=json.loads((P/'research/confirmation_selection_v036.json').read_text());check('scientific_choice_pinned',sha(P/'research/confirmation_selection_v036.json')=='ec2feebb872eaaf963ca27a1c1f61c23af0f6f1aac24b3d4a39986100b8f54ec')
    main=code(P/'confirmation_v036.ipynb');ns={'__name__':'confirmation_v036'};exec(compile(main,'confirmation_v036.ipynb','exec'),ns)
    base=json.loads((P/'research/confirmation_protocol_v014.json').read_text());sources=dict(base['source_code_hashes'])
    for name in ['postprocessing_components_v016.ipynb','laplace_components_v029.ipynb']:
        sources[name]=hashlib.sha256(code(P/'research'/name).encode()).hexdigest()
    for name,h in sources.items():
        text=code(P/'research'/name);check('component_pin:'+name,hashlib.sha256(text.encode()).hexdigest()==h);exec(compile(text,name,'exec'),ns)
    helper=code(P/'controlled_experiments_v013.ipynb');check('scoring_helper_pin',hashlib.sha256(helper.encode()).hexdigest()=='56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a');exec(compile(helper,'controlled_experiments_v013.ipynb','exec'),ns)
    selected=ns['_sequence_device'](device);torch.set_num_threads(4);ns['seed_sequence'](20261102);check('requested_device',selected.type==device)
    check('production_fit_dispatch_has_no_validation_argument',not any('val' in k for k in inspect.signature(ns['v036_fit_dispatch']).parameters))
    check('F_prediction_lock_precedes_F_truth_load',main.index("out/'predictions_locked_before_F_pressure.json'")<main.index("truth=v036_load_selected_labels"))
    check('choice_and_known_split_lock_precede_fit_labels',main.index("v036_immutable_json(out/'locked_before_F.json'")<main.index('fit_labels=v036_load_selected_labels'))
    # Execute the actual production split assignment together with the actual
    # member-identity assignment. This catches producer/consumer schema drift
    # before model allocation or any approved CSV/pressure read.
    import ast
    tree=ast.parse(main);driver=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name=='run_confirmation_v036')
    split_node=next(n for n in ast.walk(driver) if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='split' for t in n.targets))
    identity_node=next(n for n in ast.walk(driver) if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='identity' for t in n.targets))
    schema=dict(ns,afit=np.arange(9),aval=np.arange(9,11),fit=np.arange(6),val=np.arange(6,9),bids=np.arange(100,111))
    exec(compile(ast.fix_missing_locations(ast.Module(body=[split_node],type_ignores=[])),'actual_production_split_assignment','exec'),schema)
    split=schema['split'];check('production_split_schema_all_F_fields',all(set(split[k])=={'F_fit','F_validation'} for k in ['breath_ids','group_hashes','counts']) and set(split['indices'])=={'A_fit','A_validation','F_fit','F_validation'})
    for member,recipe in choice['required_members'].items():
        schema.update(selection='synthetic_lock',name=member,recipe=recipe,sources=sources)
        exec(compile(ast.fix_missing_locations(ast.Module(body=[identity_node],type_ignores=[])),'actual_production_member_identity','exec'),schema)
        check('production_split_consumed_by_identity:'+member,schema['identity']['training_group_sha256']==ns['v013_ah'](schema['bids'][schema['fit']]) and schema['identity']['recipe']==recipe)
    check('production_split_exact_synthetic_membership',split['indices']['F_fit']==list(range(6)) and split['indices']['F_validation']==list(range(6,9)) and split['breath_ids']['F_fit']==list(range(100,106)))
    # Only a selected breath has parseable labels: both original-Aval and Fval
    # can be poisoned without affecting this Ffit-only loader.
    toy=O/'toy.csv'
    with toy.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['breath_id','pressure']);w.writeheader()
        for bid in [9,2,17]:
            for j in range(80):w.writerow(dict(breath_id=bid,pressure=str(j/10) if bid in [2,9] else 'POISON'))
    got=ns['v036_load_selected_labels'](toy,[2,9]);check('selected_target_loader_excludes_poisoned_other_breaths',got.shape==(2,80) and np.array_equal(got[0],np.arange(80,dtype=np.float32)/10))
    check('selected_target_loader_preserves_requested_group_order',np.array_equal(got[0],got[1]))
    # Real fixed-budget dispatch, full seven architectures, synthetic data only.
    import pandas as pd
    rng=np.random.default_rng(20263602);steps=np.arange(80,dtype=np.float32)[None,:];u=(8+2*np.sin(steps/7)+.3*np.arange(8)[:,None]).astype(np.float32);phase=np.broadcast_to(steps>=31,(8,80)).astype(np.int8).copy()
    frame=pd.DataFrame(dict(id=np.arange(1,641,dtype=np.int64),breath_id=np.repeat(np.arange(1,9,dtype=np.int32),80),R=np.repeat([5,20,50,5,20,50,5,20],80).astype(np.int8),C=np.repeat([10,20,50,20,50,10,50,10],80).astype(np.int8),time_step=np.tile(np.arange(80,dtype=np.float32)*.025,8),u_in=u.ravel(),u_out=phase.ravel()))
    raw,names,feature_bids=ns['make_sequence_features'](frame,'combined');check('synthetic_fixture_uses_real47feature_pipeline',raw.shape==(8,80,47) and len(names)==47 and 'dt' in names)
    y=(3+.13*u+.07*steps+.1*np.arange(8)[:,None]+.03*rng.normal(size=(8,80))).astype(np.float32);summaries={}
    class ProbeInterruption(RuntimeError):pass
    def interrupt(epoch):
        if epoch['epoch']==1:raise ProbeInterruption('synthetic saved-state interruption')
    for name,original in choice['required_members'].items():
        recipe=copy.deepcopy(original);recipe['selected_epochs']=2;recipe['lr_schedule']=[.0015,.0005];recipe['train_cfg'].update(max_epochs=2,batch_size=2,min_delta=0.,scheduler='replay',frozen_lr_schedule=recipe['lr_schedule'],max_seconds=None)
        checkpoint=O/(name+'.pt');kwargs=dict(feature_names=names,device=device,resume=False,source_fingerprints=sources)
        report=ns['v036_fit_dispatch'](name,recipe,raw[:6],y[:6],phase[:6],checkpoint,**kwargs)
        check(name+':fixed_two_epochs_no_validation',report['completed'] and report['executed_epochs']==2 and report['training_breaths']==6 and not report['validation_used'] and report['executed_lr_schedule']==recipe['lr_schedule'])
        check(name+':training_only_statistics',report['preprocessing']['fitted_training_breaths']==6)
        predicted=ns['v036_predict_dispatch'](name,recipe,checkpoint,raw[6:],device=device,source_fingerprints=sources,batch_size=2)
        check(name+':native_finite_inference',predicted.shape==(2,80) and np.isfinite(predicted).all())
        if name in ['memory','capacity_laplace_curve']:
            second=O/(name+'-resume.pt')
            try:ns['v036_fit_dispatch'](name,recipe,raw[:6],y[:6],phase[:6],second,**kwargs,on_epoch=interrupt)
            except ProbeInterruption:pass
            else:raise AssertionError('Synthetic interruption not exercised')
            resumed=ns['v036_fit_dispatch'](name,recipe,raw[:6],y[:6],phase[:6],second,**dict(kwargs,resume=True));aa=torch.load(checkpoint,map_location='cpu',weights_only=False);bb=torch.load(second,map_location='cpu',weights_only=False)
            check(name+':exact_fixed_budget_resume',equal(aa['state_dict'],bb['state_dict']) and equal(aa['optimizer_state'],bb['optimizer_state']) and equal(aa['rng_state'],bb['rng_state']) and resumed['completed'])
            del aa,bb
        summaries[name]=dict(checkpoint_sha256=sha(checkpoint),parameter_count=report['parameter_count'],epochs=2,validation_used=False)
        gc.collect()
        if selected.type=='cuda':torch.cuda.empty_cache()
    check('all_seven_factories_exercised',set(summaries)==set(choice['required_members']) and len(summaries)==7)
    # Input-only postprocessing and native MAE/whole-breath-bootstrap semantics.
    post=ns['selected_postprocessing_v016'];predictions={k:np.array([0.,1.,2.],np.float64)+i*.01 for i,k in enumerate(choice['required_members'])};grid=np.arange(-3,5,dtype=np.float64)
    check('fixed_grid_median_policies_finite',np.isfinite(post({k:predictions[k] for k in choice['winner_weights']},choice['winner_weights'],grid)).all())
    # Inherit actual full-capacity runtime/resume/memory/lease checks. This also
    # measures full-batch time before ordinary-lease confirmation stages begin.
    p=P/'research/capacity_preflight_v035.ipynb';text=code(p);check('full_capacity_preflight_pinned',hashlib.sha256(text.encode()).hexdigest()=='420928745a182d0f4f991eff0cc3088bcc7bacfe4fbd3809e868edd5ff181260');cap={};exec(compile(text,str(p),'exec'),cap);capacity=cap['run_capacity_preflight_v035'](P,O/'capacity',device=device,deep_spec=choice['required_members']['capacity_laplace_curve']['model_spec'])
    checks.update({'capacity_'+k:v for k,v in capacity['checks'].items()});projected=None
    if device=='cuda':
        factor=capacity['lease_budget']['new_full_step_seconds']/.22183649241924286
        projected=dict(first_five=14496.243142550346*factor*1.05,remaining_two=(6769.328450953122+11955.665470773354*(190/200)*(54323/61114))*factor*1.05)
        check('both_ordinary_confirmation_stages_fit_30minute_lease_reserve',all(t<=19800 for t in projected.values()))
    result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),checks=checks,check_count=len(checks),source_code_hashes=sources,actual_cuda=device=='cuda',exact_resume=True,native_inference=True,cold_confirmation_contract=True,projected_stage_seconds=projected,synthetic_only=True,raw_approved_csv_reads=0,approved_data_fits=0,summaries=summaries,capacity_budget=capacity['lease_budget'],runtime=dict(torch=str(torch.__version__),numpy=np.__version__,cuda=torch.version.cuda,device=device))
    (O/'proof.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result,indent=2));return result
